# Gemfinder — Gamma Fit Evaluation & Failure Analysis


## Step 1 install dependency

In [11]:
%pip install numpy pandas scipy matplotlib

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [12]:
from pathlib import Path
for name in ["Data/RY_24.csv", "Data/RY25_PreMay10.csv", "Data/RY25_PostMay10.csv", "P_matrices_long.csv", "gamma_fit_evaluation.py", "analyse_real_instances.py"]:
    print(f"{name}: {Path(name).exists()}")

Data/RY_24.csv: False
Data/RY25_PreMay10.csv: False
Data/RY25_PostMay10.csv: False
P_matrices_long.csv: False
gamma_fit_evaluation.py: True
analyse_real_instances.py: True


## Step 3 check label/code

In [14]:
import pandas as pd
from pathlib import Path
for filename in ["RY_24.csv", "RY25_PreMay10.csv", "RY25_PostMay10.csv"]:
    path = Path("data") / filename
    if not path.exists():
        print("MISSING:", path)
        continue
    df = pd.read_csv(path, low_memory=False)
    print("\n", filename, "rows:", len(df))
    for q in ["ry10", "ry11", "ry13", "ry14"]:
        if q in df:
            print(q, df[q].value_counts(dropna=False).head(12).to_dict())


 RY_24.csv rows: 149387
ry10 {3: 57916, 4: 51549, 2: 32813, 1: 7109}
ry11 {2: 54419, 3: 40144, 1: 29006, 4: 25818}
ry13 {4: 103120, 3: 31553, 2: 11693, 1: 3021}
ry14 {4: 61471, 3: 53903, 2: 25986, 1: 8027}

 RY25_PreMay10.csv rows: 114770
ry10 {3: 44819, 4: 41348, 2: 23897, 1: 4706}
ry11 {2: 42268, 3: 31610, 1: 21071, 4: 19821}
ry13 {4: 81555, 3: 23078, 2: 8193, 1: 1944}
ry14 {4: 48899, 3: 41574, 2: 19046, 1: 5251}

 RY25_PostMay10.csv rows: 71063
ry10 {3: 28537, 4: 25995, 2: 14094, 1: 2437}
ry11 {2: 25278, 3: 20780, 4: 12601, 1: 12404}
ry13 {4: 51600, 3: 14089, 2: 4366, 1: 1008}
ry14 {4: 29553, 3: 26784, 2: 11542, 1: 3184}


## Step 4 check wether P-matrix match question


In [15]:
from pathlib import Path

p = Path("P.RData")

print("File exists:", p.exists())

if p.exists():
    print("File size:", p.stat().st_size, "bytes")

File exists: True
File size: 2194 bytes


## Step 6 — validate gamma evaluate 

In [16]:
import numpy as np
from gamma_fit_evaluation import evaluate_gamma_fit
rng = np.random.default_rng(123)
test_draws = rng.gamma(shape=2.0, scale=0.5, size=10000)
print(evaluate_gamma_fit(test_draws))

{'shape': 2.046206624229036, 'rate': 2.0249376611061245, 'ks_stat': 0.02568270882922602, 'empirical_q95': 2.35772977242837, 'gamma_q95': 2.3796308327840965, 'q95_error': 0.009289046018691583, 'tail_error': 0.0018511201961210408, 'zero_fraction': 0.0, 'n_train': 7000, 'n_test': 3000}


In [19]:
for question, value in P.items():
    matrix = np.asarray(value, dtype=float)

    if matrix.shape != (4, 4):
        continue

    row_ok = np.allclose(matrix.sum(axis=1), 1, atol=0.005)
    col_ok = np.allclose(matrix.sum(axis=0), 1, atol=0.005)

    print(
        question,
        "| rows:", row_ok,
        "| columns:", col_ok
    )

ry2 | rows: True | columns: False
ry3 | rows: False | columns: False
ry4 | rows: False | columns: False
ry5 | rows: False | columns: False
sun5 | rows: False | columns: False
ry6 | rows: False | columns: False
ry7 | rows: False | columns: False
ry9 | rows: False | columns: False
ry8 | rows: False | columns: False
ry10 | rows: False | columns: False
ry11 | rows: False | columns: False
ry12 | rows: False | columns: False
ry13 | rows: False | columns: False
ry14 | rows: False | columns: False
ry15 | rows: True | columns: False
sun4 | rows: False | columns: False
sun3 | rows: False | columns: False
sun1 | rows: True | columns: False
ry16 | rows: False | columns: False
ry17 | rows: True | columns: False
ry18 | rows: True | columns: False
ry19 | rows: False | columns: False
ry20 | rows: True | columns: False
ry21 | rows: True | columns: False
ry22 | rows: False | columns: False
tru1 | rows: False | columns: False
tru2 | rows: False | columns: False
for1 | rows: False | columns: False
for2 | 

In [20]:
import numpy as np
import pandas as pd

for question in ["ry2", "ry3", "ry10", "ry11"]:
    matrix = np.asarray(P[question], dtype=float)

    print("\n==========", question, "==========")
    print("Shape:", matrix.shape)

    display(pd.DataFrame(matrix).round(6))

    print("Row sums:", np.round(matrix.sum(axis=1), 6))
    print("Column sums:", np.round(matrix.sum(axis=0), 6))
    print("Min:", matrix.min())
    print("Max:", matrix.max())


========== ry2 ==========
Shape: (4, 4)


,0,1,2,3
0,0.65,0.15,0.12,0.08
1,0.08,0.51,0.31,0.10
2,0.01,0.10,0.58,0.31
3,0.00,0.01,0.08,0.91


Row sums: [1. 1. 1. 1.]
Column sums: [0.74 0.77 1.09 1.4 ]
Min: 0.0
Max: 0.91

========== ry3 ==========
Shape: (4, 4)


,0,1,2,3
0,0.57,0.28,0.06,0.09
1,0.08,0.59,0.26,0.07
2,0.01,0.15,0.55,0.28
3,0.01,0.03,0.19,0.78


Row sums: [1.   1.   0.99 1.01]
Column sums: [0.67 1.05 1.06 1.22]
Min: 0.01
Max: 0.78

========== ry10 ==========
Shape: (4, 4)


,0,1,2,3
0,0.50,0.41,0.04,0.04
1,0.06,0.59,0.28,0.07
2,0.01,0.17,0.66,0.16
3,0.01,0.06,0.23,0.70


Row sums: [0.99 1.   1.   1.  ]
Column sums: [0.58 1.23 1.21 0.97]
Min: 0.01
Max: 0.7

========== ry11 ==========
Shape: (4, 4)


,0,1,2,3
0,0.51,0.36,0.09,0.04
1,0.07,0.60,0.25,0.08
2,0.03,0.26,0.49,0.21
3,0.08,0.09,0.29,0.54


Row sums: [1.   1.   0.99 1.  ]
Column sums: [0.69 1.31 1.12 0.87]
Min: 0.03
Max: 0.6


## Step 7 — analysis

In [28]:
!python analyse_real_instances.py

ry10__RY_24__vs__RY25_PreMay10 evaluated_proxy
ry10__RY_24__vs__RY25_PostMay10 evaluated_proxy
ry10__RY25_PreMay10__vs__RY25_PostMay10 evaluated_proxy
ry11__RY_24__vs__RY25_PreMay10 evaluated_proxy
ry11__RY_24__vs__RY25_PostMay10 evaluated_proxy
ry11__RY25_PreMay10__vs__RY25_PostMay10 evaluated_proxy
ry13__RY_24__vs__RY25_PreMay10 evaluated_proxy
ry13__RY_24__vs__RY25_PostMay10 evaluated_proxy
ry13__RY25_PreMay10__vs__RY25_PostMay10 evaluated_proxy
ry14__RY_24__vs__RY25_PreMay10 evaluated_proxy
ry14__RY_24__vs__RY25_PostMay10 evaluated_proxy
ry14__RY25_PreMay10__vs__RY25_PostMay10 evaluated_proxy
Saved results to /home/bf9c57b3-8ba4-42f8-ac53-fc04dfaf119d/Untitled Folder/capstone/results


In [25]:
out = Path("results/gamma_failure_analysis.csv")
if out.exists():
    result = pd.read_csv(out)
    display(result.head(10))
else:
    print("Not generated yet: complete Step 7 first")

,instance_id,question,group_x,group_y,n,m,excluded_x,excluded_y,size_ratio,x_counts,...,rate,ks_stat,empirical_q95,gamma_q95,q95_error,tail_error,zero_fraction,n_train,n_test,status
0,ry10__RY25_PreMay10__vs__RY25_PostMay10,ry10,RY25_PreMay10,RY25_PostMay10,114770,71063,0,0,1.615046,"[4706, 23897, 44819, 41348]",...,22516.715687,0.039208,0.000845,0.000857,0.014435,0.006371,0.0,7000,3000,evaluated_proxy
1,ry10__RY_24__vs__RY25_PostMay10,ry10,RY_24,RY25_PostMay10,149387,71063,0,0,2.102177,"[7109, 32813, 57916, 51549]",...,23840.192618,0.018346,0.003182,0.003200,0.005795,0.005242,0.0,7000,3000,evaluated_proxy
2,ry10__RY_24__vs__RY25_PreMay10,ry10,RY_24,RY25_PreMay10,149387,114770,0,0,1.301621,"[7109, 32813, 57916, 51549]",...,32767.326907,0.014263,0.001107,0.001111,0.003986,0.002406,0.0,7000,3000,evaluated_proxy


## 重要限制
这套程序读取的 **X/Y 确实来自真实问卷 CSV**，但它把观察到的回答类别当作潜在学生类型来模拟 retest，属于 **proxy 假设**。原 R 代码的潜在类型由模拟生成，二者并非已证明等价。

因此在团队确认真实 instance 模拟方式之前，输出只能标为 **survey-derived exploratory Gamma diagnostics**，不能声称已经验证了真实数据中的 Gamma failure。

In [26]:
import pandas as pd
from pathlib import Path

files = [
    "data/RY_24.csv",
    "data/RY25_PreMay10.csv",
    "data/RY25_PostMay10.csv"
]

for file in files:

    if not Path(file).exists():
        print("File not found:", file)
        continue

    df = pd.read_csv(file, low_memory=False)

    print("\nDATASET:", file)

    # Find all ry questions
    ry_questions = [
        col for col in df.columns
        if col.startswith("ry") and col[2:].isdigit()
    ]

    for question in ry_questions:

        values = pd.to_numeric(
            df[question],
            errors="coerce"
        ).dropna()

        unique_values = sorted(values.unique().tolist())

        print(question, unique_values[:20])


DATASET: data/RY_24.csv
ry10 [1, 2, 3, 4]
ry11 [1, 2, 3, 4]
ry13 [1, 2, 3, 4]
ry14 [1, 2, 3, 4]
ry15 [1, 2, 3, 4]
ry16 [1, 2, 3, 4]
ry17 [1, 2, 3, 4]
ry18 [1, 2, 3, 4]
ry2 [1, 2, 3, 4]
ry20 [1.0, 2.0, 3.0, 4.0]
ry21 [1, 2, 3, 4]
ry22 [1, 2, 3, 4]
ry25 [1.0, 2.0, 3.0, 4.0]
ry26 [1, 2, 3, 4]
ry27 [1, 2, 3, 4]
ry28 [1, 2, 3, 4]
ry29 [1, 2, 3, 4]
ry3 [1, 2, 3, 4]
ry30 [1, 2, 3, 4]
ry31 [1, 2, 3, 4]
ry32 [1.0, 2.0, 3.0, 4.0]
ry35 [1, 2, 3, 4]
ry36 [1, 2, 3, 4]
ry37 [1, 2, 3, 4]
ry38 [1, 2, 3, 4]
ry39 [1, 2, 3, 4]
ry4 [1, 2, 3, 4]
ry40 [1.0, 2.0, 3.0, 4.0]
ry42 [1.0, 2.0, 3.0, 4.0]
ry43 [1.0, 2.0, 3.0, 4.0]
ry5 [1, 2, 3, 4]
ry6 [1, 2, 3, 4]
ry7 [1.0, 2.0, 3.0, 4.0]
ry8 [1, 2, 3, 4]
ry9 [1.0, 2.0, 3.0, 4.0]

DATASET: data/RY25_PreMay10.csv
ry2 [1, 2, 3, 4]
ry3 [1, 2, 3, 4]
ry4 [1, 2, 3, 4]
ry5 [1, 2, 3, 4]
ry6 [1, 2, 3, 4]
ry7 [1.0, 2.0, 3.0, 4.0]
ry8 [1, 2, 3, 4]
ry9 [1.0, 2.0, 3.0, 4.0]
ry10 [1, 2, 3, 4]
ry11 [1, 2, 3, 4]
ry13 [1, 2, 3, 4]
ry14 [1, 2, 3, 4]
ry15 [1, 2, 3, 4]
ry35 [1, 2, 3,

In [1]:
%pip install rdata

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [3]:
import rdata
from pathlib import Path

p_path = Path("P.RData")

print("File exists:", p_path.exists())

P = rdata.read_rds(str(p_path))

print("Successfully loaded P!")
print("Object type:", type(P))

File exists: True
Successfully loaded P!
Object type: <class 'dict'>


In [2]:
!pip install rdata
import rdata
from pathlib import Path

p_path = Path("P.RData")

print("File exists:", p_path.exists())

P = rdata.read_rda(str(p_path))

print("Object type:", type(P))
print("Object names:", list(P.keys()))

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
File exists: True
Object type: <class 'dict'>
Object names: ['ry2', 'ry3', 'ry4', 'ry5', 'sun5', 'ry6', 'ry7', 'ry9', 'ry8', 'ry10', 'ry11', 'ry12', 'ry13', 'ry14', 'ry15', 'sun4', 'sun3', 'sun1', 'ry16', 'ry17', 'ry18', 'ry19', 'ry20', 'ry21', 'ry22', 'tru1', 'tru2', 'for1', 'for2', 'ry23', 'ry24', 'ry25', 'ry26', 'ry27', 'ry28', 'ry29', 'ry30', 'ry31', 'sun11', 'sun12', 'cyb1', 'ph1', 'ph2', 'ph3', 'ph4', 'phq3', 'phq4', 'phq5', 'phq7', 'cop1', 'cop2', 'cop3', 'cop4', 'chs1', 'chs2', 'chs3', 'chs4', 'chs5', 'chs6']


/home/bf9c57b3-8ba4-42f8-ac53-fc04dfaf119d/.local/lib/python3.12/site-packages/rdata/parser/_parser.py:1280: UserWarning: Unknown file type: assumed RDS
  warnings.warn("Unknown file type: assumed RDS")  # noqa: B028
/home/bf9c57b3-8ba4-42f8-ac53-fc04dfaf119d/.local/lib/python3.12/site-packages/rdata/parser/_parser.py:1283: UserWarning: Wrong extension .rda for file in RDS format
  warnings.warn(  # noqa: B028
